In [ ]:
!pip install openai openpyxl tqdm -q

In [ ]:
from google.colab import files
uploaded = files.upload()   # Select Visualized interpretability (13).xlsx
INPUT_XLSX = list(uploaded.keys())[0]
print(f"Uploaded: {INPUT_XLSX}")

In [ ]:
import os
from google.colab import userdata

# Get the key from Secrets storage.
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]

# Other settings
OUTPUT_XLSX = "Visualized interpretability (13).xlsx"
CACHE_FILE = "semantic_cache.json"
COL_SUBJECT = 15
COL_PURPOSE = 16
N_WORKERS = 10
N_SAMPLE = 0

In [ ]:
"""
Figure Caption Semantic Extractor
-----------------------------------
Extracts semantics from figure captions in column N for each figure,
adding new columns while maintaining the existing rows (papers) of the xlsx:
  - col O: visualization_subject  (per figure, collected in one cell)
  - col P: visualization_purpose  (per figure, collected in one cell)

Usage:
  python extract_semantic.py --api-key sk-...  [--sample 20] [--workers 10]

Requirements:
  pip install openai openpyxl tqdm
"""

import argparse, json, os, re, time
import openpyxl
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
from openai import OpenAI

# ─── Path Settings ───────────────────────────────────────────────────────────
INPUT_XLSX  = 'Visualized interpretability (12).xlsx'
OUTPUT_XLSX = 'Visualized interpretability (13).xlsx'   # Final version with added columns
CACHE_FILE  = 'semantic_cache.json'                     # For resuming after interruption

COL_SUBJECT = 15   # Column O: visualization_subject (per figure)
COL_PURPOSE = 16   # Column P: visualization_purpose (per figure)

# ─── Prompt ──────────────────────────────────────────────────────────────────
SYSTEM_PROMPT = """You are an expert in interpretability and explainability research.

Given a figure caption from an interpretability paper, extract the following:

0. extractable: true | false
   - false if the caption is too short, a section title fragment, a bare label
     (e.g. "Parameter analysis.", "thousand steps", "Lexical Sensitivity 5.1.1"),
     or otherwise lacks enough information to infer subject or purpose.
   - If false, set subject and purpose to empty strings and stop.
   - If true, proceed to extract subject and purpose.

1. visualization_subject (5–15 words):
   A concise noun phrase describing WHAT is being measured or visualized.
   Focus on the concrete object/phenomenon in the figure (e.g., a specific metric,
   model component, data distribution, or experimental variable).

2. visualization_purpose (10–20 words):
   A concise phrase capturing WHY this figure exists — what insight, claim,
   or comparison it is meant to convey to the reader.

3. confidence: "high" | "medium" | "low"
   - high: caption clearly describes the figure content
   - medium: some ambiguity, but enough context to extract
   - low: extractable but caption is vague or incomplete

Rules:
- Be specific, not generic. Avoid vague phrases like "model performance" or "experimental results".
- Use domain-appropriate terms (e.g., "attention head", "residual stream", "SAE feature",
  "probing accuracy", "activation patching", "logit lens").
- Do NOT paraphrase the caption. Abstract the key meaning.
- If the caption describes multiple panels, focus on the shared theme.

Examples:

Caption: "Token-level feature activation heatmaps. Darker highlights indicate higher
activation values of specific SAE features."
Viz type: Heatmap / Matrix
→ subject: "SAE feature activation intensity across input tokens"
→ purpose: "show which token positions most strongly activate specific learned sparse features"
→ confidence: "high"

Caption: "t-SNE visualization of Meta-Llama-3.1-70B last-token activations from the 1st,
50th, and 80th layers, using 11 continuation-style templates across 50 elements."
Viz type: Scatter Plot / Projection
→ subject: "LLM internal representation geometry by layer and prompt template"
→ purpose: "demonstrate how activation space structure evolves across layers and varies by input attribute"
→ confidence: "high"

Caption: "Overlaps and differences in ISNs distribution across same-type and different-type
instructions on LLaMA-2-Chat-7B."
Viz type: Heatmap / Matrix
→ subject: "instruction-specific neuron overlap across instruction types and model sizes"
→ purpose: "reveal that same-type instructions share more active neurons than different-type instructions"
→ confidence: "high"

Caption: "Impact of τ on Behavior Steering. Effect of varying τ, which controls the
sparsity of SAS vectors, on behavior modulation."
Viz type: Line Chart
→ subject: "sparsity hyperparameter effect on steering vector behavior modulation"
→ purpose: "show how sparsity level trades off between steering precision and intervention strength"
→ confidence: "high"

Caption: "Parameter analysis."
Viz type: Bar Chart
→ extractable: false
→ subject: ""
→ purpose: ""
→ confidence: "low"

Caption: "thousand steps"
Viz type: Line Chart
→ extractable: false
→ subject: ""
→ purpose: ""
→ confidence: "low"
"""

RESPONSE_SCHEMA = {
    "type": "object",
    "properties": {
        "extractable": {"type": "boolean"},
        "visualization_subject": {"type": "string"},
        "visualization_purpose": {"type": "string"},
        "confidence": {"type": "string", "enum": ["high", "medium", "low"]}
    },
    "required": ["extractable", "visualization_subject", "visualization_purpose", "confidence"],
    "additionalProperties": False
}


# ─── Parsing ──────────────────────────────────────────────────────────────────
FIG_BLOCK = re.compile(
    r'\[Figure\s+([^\]]+)\]\s+([^\n]+)\n(.*?)(?=\[Figure\s|\Z)',
    re.DOTALL
)

def parse_figures(xlsx_path):
    wb = openpyxl.load_workbook(xlsx_path)
    ws = wb['Taxonomy']
    figures = []
    for r in range(2, ws.max_row + 1):
        n_val = ws.cell(r, 14).value
        url   = ws.cell(r, 2).value or ''
        if not n_val:
            continue
        for m in FIG_BLOCK.finditer(str(n_val)):
            fig_label = m.group(1).strip()
            viz_type  = m.group(2).strip()
            caption   = m.group(3).strip()
            if len(caption) >= 10:
                figures.append({
                    'row':       r,
                    'fig_label': fig_label,
                    'viz_type':  viz_type,
                    'caption':   caption[:800],
                    'url':       str(url)[:100],
                })
    return figures


# ─── GPT-4o Call ──────────────────────────────────────────────────────────────
def extract_one(client, fig: dict, max_retries=3):
    key = f"{fig['row']}_{fig['fig_label']}"
    user_msg = (
        f"Figure type: {fig['viz_type']}\n\n"
        f"Caption:\n{fig['caption']}"
    )
    for attempt in range(max_retries):
        try:
            resp = client.chat.completions.create(
                model="gpt-4o-2024-08-06",
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user",   "content": user_msg},
                ],
                response_format={
                    "type": "json_schema",
                    "json_schema": {
                        "name": "figure_semantic",
                        "strict": True,
                        "schema": RESPONSE_SCHEMA
                    }
                },
                temperature=0,
                max_tokens=200,
            )
            result = json.loads(resp.choices[0].message.content)
            return key, {**fig, **result}
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)
            else:
                return key, {**fig,
                    'visualization_subject': 'ERROR',
                    'visualization_purpose': str(e),
                    'confidence': 'low'}


# ─── Save: Add columns O, P to existing xlsx (maintain row = paper unit) ────
CONF_COLOR = {'high': 'E2EFDA', 'medium': 'FFF9C4', 'low': 'FFE0B3'}

def build_row_cells(results_by_row):
    """
    Aggregate per-figure results to per-paper row units.
    Returns: {row_num: {'subject_text': str, 'purpose_text': str, 'worst_conf': str}}
    """
    from collections import defaultdict
    rows = defaultdict(list)
    for v in results_by_row.values():
        rows[v['row']].append(v)

    out = {}
    conf_rank = {'high': 0, 'medium': 1, 'low': 2}
    for row_num, figs in rows.items():
        figs_sorted = sorted(figs, key=lambda x: x['fig_label'])
        subj_lines = []
        purp_lines = []
        for f in figs_sorted:
            label = f"[Figure {f['fig_label']}]"
            if not f.get('extractable', True):
                subj_lines.append(f"{label} (insufficient caption)")
                purp_lines.append(f"{label} (insufficient caption)")
            else:
                subj_lines.append(f"{label} {f.get('visualization_subject', '')}")
                purp_lines.append(f"{label} {f.get('visualization_purpose', '')}")
        worst = max(figs_sorted,
                    key=lambda x: conf_rank.get(x.get('confidence', 'low'), 2))
        out[row_num] = {
            'subject_text': '\n'.join(subj_lines),
            'purpose_text': '\n'.join(purp_lines),
            'worst_conf':   worst.get('confidence', 'low'),
        }
    return out

def save_results(results, input_path, out_path):
    from openpyxl.styles import PatternFill, Alignment, Font, Border, Side

    wb = openpyxl.load_workbook(input_path)
    ws = wb['Taxonomy']

    bold = Font(bold=True)
    thin = Side(style='thin', color='CCCCCC')
    bdr  = Border(left=thin, right=thin, top=thin, bottom=thin)
    wrap = Alignment(wrap_text=True, vertical='top')

    # Add columns O, P to header row
    for col, header in [(COL_SUBJECT, '⑪ Visualization Subject'),
                        (COL_PURPOSE, '⑫ Visualization Purpose')]:
        cell = ws.cell(1, col, header)
        cell.font  = bold
        cell.fill  = PatternFill('solid', fgColor='BDD7EE')
        cell.border = bdr
        col_letter = openpyxl.utils.get_column_letter(col)
        ws.column_dimensions[col_letter].width = 45

    row_data = build_row_cells(results)

    for row_num, data in row_data.items():
        fill = PatternFill('solid', fgColor=CONF_COLOR.get(data['worst_conf'], 'FFE0B3'))
        for col, text in [(COL_SUBJECT, data['subject_text']),
                          (COL_PURPOSE, data['purpose_text'])]:
            cell = ws.cell(row_num, col, text)
            cell.fill      = fill
            cell.border    = bdr
            cell.alignment = wrap

    import shutil
    shutil.copy2(input_path, out_path)          # Copy existing file then overwrite
    wb.save(out_path)
    print(f"Saved → {out_path}  ({len(row_data)} paper rows updated)
")


# ─── Main ────────────────────────────────────────────────────────────────────
def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--api-key', required=True, help='OpenAI API key')
    parser.add_argument('--sample',  type=int, default=0,
                        help='Process only N figures (0 = all)')
    parser.add_argument('--workers', type=int, default=10,
                        help='Concurrent API calls (default 10)')
    parser.add_argument('--input',  default=INPUT_XLSX)
    parser.add_argument('--output', default=OUTPUT_XLSX)
    args = parser.parse_args()

    client  = OpenAI(api_key=args.api_key)
    figures = parse_figures(args.input)
    print(f"Parsed {len(figures)} figures from {args.input}")

    # Sample mode
    if args.sample > 0:
        import random
        figures = random.sample(figures, min(args.sample, len(figures)))
        print(f"Sample mode: processing {len(figures)} figures")

    # Load cache (resume after interruption)
    cache = {}
    if os.path.exists(CACHE_FILE):
        with open(CACHE_FILE) as f:
            cache = json.load(f)
        print(f"Loaded {len(cache)} cached results")

    todo = [f for f in figures
            if f"{f['row']}_{f['fig_label']}" not in cache]
    print(f"Remaining to process: {len(todo)}")

    # API Call
    with ThreadPoolExecutor(max_workers=args.workers) as pool:
        futures = {pool.submit(extract_one, client, fig): fig for fig in todo}
        with tqdm(total=len(futures), desc='Extracting') as pbar:
            for fut in as_completed(futures):
                key, result = fut.result()
                cache[key] = result
                pbar.update(1)
                # Save intermittently every 50 items
                if len(cache) % 50 == 0:
                    with open(CACHE_FILE, 'w') as f:
                        json.dump(cache, f, ensure_ascii=False, indent=2)

    # Final save
    with open(CACHE_FILE, 'w') as f:
        json.dump(cache, f, ensure_ascii=False, indent=2)

    # Output to the same directory as the input file
    in_dir   = os.path.dirname(os.path.abspath(args.input))
    out_path = os.path.join(in_dir, args.output)
    save_results(cache, args.input, out_path)

    # Statistics
    highs   = sum(1 for v in cache.values() if v.get('confidence') == 'high')
    mediums = sum(1 for v in cache.values() if v.get('confidence') == 'medium')
    lows    = sum(1 for v in cache.values() if v.get('confidence') == 'low')
    print(f"\nConfidence — high:{highs}  medium:{mediums}  low:{lows}")
    print(f"High+Medium rate: {(highs+mediums)/len(cache)*100:.1f}%")

In [ ]:
import random
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm   # ← Colab uses tqdm.notebook
from openai import OpenAI

client  = OpenAI(api_key=OPENAI_API_KEY)
figures = parse_figures(INPUT_XLSX)
print(f"Total figures: {len(figures)}")

# Test mode
if N_SAMPLE > 0:
    figures = random.sample(figures, min(N_SAMPLE, len(figures)))
    print(f"Sample mode: {len(figures)} figures")

# Load cache (resume after interruption)
cache = {}
if os.path.exists(CACHE_FILE):
    with open(CACHE_FILE) as f:
        cache = json.load(f)
    print(f"Cache loaded: {len(cache)} items")

todo = [f for f in figures if f"{f['row']}_{f['fig_label']}" not in cache]
print(f"Remaining tasks: {len(todo)}")

with ThreadPoolExecutor(max_workers=N_WORKERS) as pool:
    futures = {pool.submit(extract_one, client, fig): fig for fig in todo}
    with tqdm(total=len(futures), desc="Extracting") as pbar:
        for fut in as_completed(futures):
            key, result = fut.result()
            cache[key] = result
            pbar.update(1)
            if len(cache) % 50 == 0:
                with open(CACHE_FILE, "w") as f:
                    json.dump(cache, f, ensure_ascii=False)

with open(CACHE_FILE, "w") as f:
    json.dump(cache, f, ensure_ascii=False, indent=2)

save_results(cache, INPUT_XLSX, OUTPUT_XLSX)

highs   = sum(1 for v in cache.values() if v.get("confidence") == "high")
mediums = sum(1 for v in cache.values() if v.get("confidence") == "medium")
lows    = sum(1 for v in cache.values() if v.get("confidence") == "low")
print(f"\nConfidence — high:{highs}  medium:{mediums}  low:{lows}")
print(f"High+Medium rate: {(highs+mediums)/len(cache)*100:.1f}%")

In [ ]:
files.download(OUTPUT_XLSX)
files.download(CACHE_FILE)   # 나중에 재개/샘플링용으로 보관

In [ ]:
from google.colab import files
uploaded = files.upload()   # Select semantic_cache.json
CACHE_FILE = list(uploaded.keys())[0]

In [ ]:
N_SAMPLE = 100
SEED     = 42
OUTPUT   = "grading_sample.xlsx"

In [ ]:
"""
Grading Sheet Generator
-----------------------
Generates a grading Excel file by sampling 100 items from semantic_cache.json.

- Proportional sampling by confidence (maintains high/medium/low ratio)
- Includes some 'extractable=false' items separately (to verify correct skipping)
- Grading columns: Subject OK / Purpose OK / Memo

Usage:
  python make_grading_sheet.py [--cache semantic_cache.json] [--n 100]
"""

import argparse, json, random
import openpyxl
from openpyxl.styles import PatternFill, Alignment, Font, Border, Side
from openpyxl.worksheet.datavalidation import DataValidation

CACHE_FILE  = 'semantic_cache.json'
OUTPUT_FILE = 'grading_sample.xlsx'
N_SAMPLE    = 100

CONF_BG = {'high': 'E2EFDA', 'medium': 'FFF9C4', 'low': 'FFE0B3'}
GRAY    = 'F2F2F2'
HEADER  = 'D0D0D0'
BLUE    = 'BDD7EE'
RED_BG  = 'FCE4D6'   # extractable=false rows


def stratified_sample(items, n):
    """
    Samples n items while maintaining the confidence ratio.
    Includes up to 10 'extractable=false' items separately.
    """
    extractable   = [x for x in items if x.get('extractable', True)]
    not_extractable = [x for x in items if not x.get('extractable', True)]

    # extractable=false: max 10 items (for verifying correct skipping)
    n_skip  = min(10, len(not_extractable))
    skipped = random.sample(not_extractable, n_skip)

    # The remaining 90 items maintain the confidence ratio from extractable=true
    n_main  = n - n_skip
    by_conf = {'high': [], 'medium': [], 'low': []}
    for x in extractable:
        by_conf[x.get('confidence', 'low')].append(x)

    total_ext = len(extractable)
    chosen = []
    for conf, bucket in by_conf.items():
        quota = round(n_main * len(bucket) / total_ext) if total_ext else 0
        chosen.extend(random.sample(bucket, min(quota, len(bucket))))

    # Correction for rounding errors
    while len(chosen) < n_main and extractable:
        extra = random.choice(extractable)
        if extra not in chosen:
            chosen.append(extra)

    # Combine and shuffle
    combined = chosen[:n_main] + skipped
    random.shuffle(combined)
    return combined


def make_sheet(samples, out_path):
    wb = openpyxl.Workbook()
    ws = wb.active
    ws.title = 'Grading'

    # ── Styles ──────────────────────────────────────────────
    bold = Font(bold=True, size=10)
    norm = Font(size=10)
    thin = Side(style='thin', color='CCCCCC')
    bdr  = Border(left=thin, right=thin, top=thin, bottom=thin)
    wrap = Alignment(wrap_text=True, vertical='top')
    ctr  = Alignment(horizontal='center', vertical='top')

    # ── Header ────────────────────────────────────────────────
    headers = [
        ('#',               4),
        ('Row (Paper)',       8),
        ('Figure',          9),
        ('Viz Type',       18),
        ('Confidence',     11),
        ('Extractable',    11),
        ('Caption',        52),
        ('Visualization Subject (Extraction Result)',  38),
        ('Visualization Purpose (Extraction Result)',  45),
        ('Subject OK?',    12),   # Grading
        ('Purpose OK?',    12),   # Grading
        ('Memo',            22),   # Grading
    ]
    for c, (h, w) in enumerate(headers, 1):
        cell = ws.cell(1, c, h)
        cell.font   = bold
        cell.fill   = PatternFill('solid', fgColor=HEADER)
        cell.border = bdr
        cell.alignment = Alignment(horizontal='center', vertical='center',
                                   wrap_text=True)
        ws.column_dimensions[openpyxl.utils.get_column_letter(c)].width = w

    ws.row_dimensions[1].height = 32

    # ── Grading column dropdown (Y / N / ?) ────────────────
    dv = DataValidation(type='list', formula1='"Y,N,?"', allow_blank=True)
    dv.sqref = f'J2:K{N_SAMPLE + 1}'
    ws.add_data_validation(dv)

    # ── Data Rows ───────────────────────────────────────────
    for i, item in enumerate(samples, 2):
        is_ext  = item.get('extractable', True)
        conf    = item.get('confidence', 'low')

        if not is_ext:
            bg = RED_BG
        else:
            bg = CONF_BG.get(conf, 'FFE0B3')

        fill = PatternFill('solid', fgColor=bg)

        vals = [
            i - 1,                                          # #
            item.get('row', ''),                            # Row
            f"Figure {item.get('fig_label', '')}",         # Figure
            item.get('viz_type', ''),                       # Viz Type
            conf,                                           # Confidence
            'Yes' if is_ext else 'No',                      # Extractable
            item.get('caption', ''),                        # Caption
            item.get('visualization_subject', ''),          # Subject
            item.get('visualization_purpose', ''),          # Purpose
            '',                                             # Subject OK? (Grading)
            '',                                             # Purpose OK? (Grading)
            '',                                             # Memo (Grading)
        ]
        for c, v in enumerate(vals, 1):
            cell = ws.cell(i, c, v)
            cell.font      = norm
            cell.border    = bdr
            cell.fill      = fill
            if c in (1, 2, 5, 6, 10, 11):   # Center align for numbers/short columns
                cell.alignment = ctr
            else:
                cell.alignment = wrap

        ws.row_dimensions[i].height = 72   # For multi-line captions

    # ── Emphasize grading columns ──────────────────────────
    for c in (10, 11, 12):
        col_letter = openpyxl.utils.get_column_letter(c)
        for r in range(2, len(samples) + 2):
            ws.cell(r, c).fill = PatternFill('solid', fgColor='EAF4FB')

    ws.freeze_panes = 'G2'   # Freeze up to Caption

    # ── Legend Sheet ────────────────────────────────────────────
    legend = wb.create_sheet('Legend')
    legend_rows = [
        ('Color', 'Meaning'),
        ('Green (E2EFDA)', 'confidence=high — caption is sufficiently clear'),
        ('Yellow (FFF9C4)', 'confidence=medium — extracted but somewhat ambiguous'),
        ('Orange (FFE0B3)', 'confidence=low — extracted but caption is unclear'),
        ('Light Red (FCE4D6)', 'extractable=false — determined as not extractable due to insufficient caption'),
        ('', ''),
        ('Grading Criteria', ''),
        ('Y', 'Extraction result is meaningful and correct'),
        ('N', 'Extraction result is wrong or meaningless'),
        ('?', 'Difficult to judge / Ambiguous'),
        ('', ''),
        ('Goal', '90+ out of 100 for Subject OK=Y, 90+ out of 100 for Purpose OK=Y'),
    ]
    for r, (a, b) in enumerate(legend_rows, 1):
        legend.cell(r, 1, a).font = Font(bold=(r == 1 or a in ('Grading Criteria', 'Goal')))
        legend.cell(r, 2, b)
    legend.column_dimensions['A'].width = 22
    legend.column_dimensions['B'].width = 55

    wb.save(out_path)
    print(f"Saved → {out_path}  ({len(samples)} samples)")

    # ── Sample Composition Summary ─────────────────────────
    from collections import Counter
    confs = Counter(x.get('confidence') for x in samples)
    exts  = Counter(x.get('extractable', True) for x in samples)
    print(f"  confidence — {dict(confs)}")
    print(f"  extractable — True:{exts[True]}  False:{exts[False]}")


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--cache',  default=CACHE_FILE)
    parser.add_argument('--n',      type=int, default=N_SAMPLE)
    parser.add_argument('--seed',   type=int, default=42)
    parser.add_argument('--output', default=OUTPUT_FILE)
    args = parser.parse_args()

    random.seed(args.seed)

    with open(args.cache) as f:
        cache = json.load(f)

    items = list(cache.values())
    print(f"Loaded {len(items)} figures from cache")

    samples = stratified_sample(items, args.n)
    make_sheet(samples, args.output)

In [ ]:
import random, json

random.seed(SEED)

with open(CACHE_FILE) as f:
    cache = json.load(f)

items   = list(cache.values())
samples = stratified_sample(items, N_SAMPLE)
make_sheet(samples, OUTPUT)

In [ ]:
files.download(OUTPUT)


In [ ]:
!pip install openai openpyxl umap-learn hdbscan numpy pandas tqdm -q

In [ ]:
from google.colab import files
import json, os

print("Uploading semantic_cache.json")
up = files.upload()
CACHE_FILE = [k for k in up if k.endswith('.json')][0]

print("\nUploading (13).xlsx")
up2 = files.upload()
INPUT_XLSX = [k for k in up2 if k.endswith('.xlsx')][0]

OUTPUT_XLSX = INPUT_XLSX.replace('(13)', '(14)')
print(f"Output filename: {OUTPUT_XLSX}")

In [ ]:
import os
from google.colab import userdata

# Retrieve 'OPENAI_API_KEY' stored in Secrets storage.
try:
    os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
    OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]
    print("Success: Loaded saved API key.")
except userdata.SecretNotFoundError:
    print("Error: Please set 'OPENAI_API_KEY' first in the Colab 'Secrets (key icon)' menu.")
MIN_CLUSTER_SIZE = 8   # Minimum number of figures per cluster
MIN_SAMPLES      = 3   # Noise sensitivity (lower for more clusters)
UMAP_NEIGHBORS   = 15  # UMAP n_neighbors
UMAP_DIM_CLUSTER = 10  # UMAP dimensions for clustering
UMAP_DIM_VIZ     = 2   # UMAP dimensions for visualization

COL_SUBJ_CLUSTER = 17  # Column Q
COL_PURP_CLUSTER = 18  # Column R

In [ ]:
with open(CACHE_FILE) as f:
    cache = json.load(f)

all_figs = list(cache.values())
extractable = [f for f in all_figs if f.get('extractable', True)
               and f.get('visualization_subject', '').strip()]

print(f"Total figures: {len(all_figs)}")
print(f"Extractable (for clustering): {len(extractable)}")
print(f"Excluded (insufficient caption): {len(all_figs) - len(extractable)}")

subjects = [f['visualization_subject'] for f in extractable]
purposes = [f['visualization_purpose'] for f in extractable]

In [ ]:
import numpy as np
from openai import OpenAI
from tqdm.notebook import tqdm

client = OpenAI(api_key=OPENAI_API_KEY)

def embed_texts(texts, model='text-embedding-3-small', batch_size=500):
    all_embs = []
    for i in tqdm(range(0, len(texts), batch_size), desc='Embedding'):
        batch = texts[i:i+batch_size]
        resp  = client.embeddings.create(input=batch, model=model)
        all_embs.extend([d.embedding for d in resp.data])
    return np.array(all_embs)

print("Embedding subjects...")
subj_emb = embed_texts(subjects)
print("Embedding purposes...")
purp_emb = embed_texts(purposes)
print(f"Shape: {subj_emb.shape}")

In [ ]:
import umap, hdbscan

def cluster(embeddings, label=''):
    # UMAP for clustering (high-dimensional)
    reducer = umap.UMAP(n_neighbors=UMAP_NEIGHBORS,
                        n_components=UMAP_DIM_CLUSTER,
                        metric='cosine', random_state=42)
    reduced = reducer.fit_transform(embeddings)

    # HDBSCAN
    clusterer = hdbscan.HDBSCAN(min_cluster_size=MIN_CLUSTER_SIZE,
                                 min_samples=MIN_SAMPLES,
                                 metric='euclidean',
                                 cluster_selection_method='eom')
    labels = clusterer.fit_predict(reduced)

    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise    = (labels == -1).sum()
    print(f"[{label}] Clusters: {n_clusters}  Noise: {n_noise} "
          f"({n_noise/len(labels)*100:.1f}%)")

    # 2D UMAP for visualization (separate)
    viz = umap.UMAP(n_neighbors=UMAP_NEIGHBORS, n_components=UMAP_DIM_VIZ,
                    metric='cosine', random_state=42).fit_transform(embeddings)

    return labels, viz, clusterer

print("Clustering subjects...")
subj_labels, subj_viz, subj_clusterer = cluster(subj_emb, 'Subject')
print("\nClustering purposes...")
purp_labels, purp_viz, purp_clusterer = cluster(purp_emb, 'Purpose')

In [ ]:
import matplotlib.pyplot as plt

def plot_clusters(viz, labels, title):
    fig, ax = plt.subplots(figsize=(10, 7))
    unique = sorted(set(labels))
    cmap   = plt.cm.get_cmap('tab20', len(unique))
    for i, lbl in enumerate(unique):
        mask = labels == lbl
        name = 'noise' if lbl == -1 else f'C{lbl}'
        ax.scatter(viz[mask, 0], viz[mask, 1],
                   c=[cmap(i)], label=name, s=12, alpha=0.7)
    ax.set_title(title)
    ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left',
              fontsize=7, ncol=2, markerscale=2)
    plt.tight_layout()
    plt.show()

plot_clusters(subj_viz, subj_labels, 'Subject Clusters')
plot_clusters(purp_viz, purp_labels, 'Purpose Clusters')

In [ ]:
LABEL_PROMPT = """You are labeling clusters of figure subjects/purposes from interpretability papers.

Given 10–15 representative examples from one cluster, generate:
1. label (2–4 words, snake_case): a concise category name
2. description (10–15 words): what this cluster represents

Be specific to interpretability/explainability research.
Avoid generic terms like "model_analysis" or "performance_evaluation".

Good examples:
- label: "attention_head_circuits"
  description: "mechanistic analysis of how attention heads implement specific computational functions"
- label: "SAE_feature_interpretability"
  description: "evaluating the semantic meaning and monosemanticity of sparse autoencoder features"
- label: "layer_wise_probing"
  description: "probing classifiers measuring how well layers encode linguistic or factual properties"
"""

LABEL_SCHEMA = {
    "type": "object",
    "properties": {
        "label":       {"type": "string"},
        "description": {"type": "string"}
    },
    "required": ["label", "description"],
    "additionalProperties": False
}

def label_cluster(examples, cluster_id):
    examples_str = '\n'.join(f'- {e}' for e in examples[:15])
    resp = client.chat.completions.create(
        model='gpt-4o-2024-08-06',
        messages=[
            {'role': 'system', 'content': LABEL_PROMPT},
            {'role': 'user',   'content':
             f'Cluster {cluster_id} examples:\n{examples_str}'}
        ],
        response_format={
            'type': 'json_schema',
            'json_schema': {
                'name': 'cluster_label',
                'strict': True,
                'schema': LABEL_SCHEMA
            }
        },
        temperature=0, max_tokens=100
    )
    return json.loads(resp.choices[0].message.content)

def generate_labels(texts, labels, field_name):
    unique_clusters = sorted(c for c in set(labels) if c != -1)
    cluster_map = {}   # cluster_id → {label, description}

    for cid in tqdm(unique_clusters, desc=f'{field_name} 라벨 생성'):
        idxs     = np.where(labels == cid)[0]
        examples = [texts[i] for i in idxs]
        # 대표 예시: 임베딩 centroid 기준 가까운 순 (간단하게 random shuffle 후 앞 15개)
        import random; random.shuffle(examples)
        result   = label_cluster(examples, cid)
        cluster_map[cid] = result
        print(f"  C{cid} ({len(idxs)}개): {result['label']}")

    cluster_map[-1] = {'label': 'unclustered', 'description': 'noise / no clear cluster'}
    return cluster_map

print("Subject cluster label")
subj_cluster_map = generate_labels(subjects, subj_labels, 'Subject')

print("\nPurpose cluster label")
purp_cluster_map = generate_labels(purposes, purp_labels, 'Purpose')

In [ ]:
from collections import defaultdict

# Assign clusters to extractable figures
for i, fig in enumerate(extractable):
    fig['subj_cluster_id']    = int(subj_labels[i])
    fig['subj_cluster_label'] = subj_cluster_map[int(subj_labels[i])]['label']
    fig['purp_cluster_id']    = int(purp_labels[i])
    fig['purp_cluster_label'] = purp_cluster_map[int(purp_labels[i])]['label']

# Assign 'unclustered' to extractable=false figures
for fig in all_figs:
    if 'subj_cluster_label' not in fig:
        fig['subj_cluster_label'] = 'unclustered'
        fig['purp_cluster_label'] = 'unclustered'

# Aggregate by row (paper)
def build_cluster_cells(figs):
    rows = defaultdict(list)
    for f in figs:
        rows[f['row']].append(f)

    out = {}
    for row_num, row_figs in rows.items():
        sorted_figs = sorted(row_figs, key=lambda x: x['fig_label'])
        subj_lines  = [f"[Figure {f['fig_label']}] {f['subj_cluster_label']}"
                       for f in sorted_figs]
        purp_lines  = [f"[Figure {f['fig_label']}] {f['purp_cluster_label']}"
                       for f in sorted_figs]
        out[row_num] = {
            'subj_text': '\n'.join(subj_lines),
            'purp_text': '\n'.join(purp_lines),
        }
    return out

row_clusters = build_cluster_cells(all_figs)

# Save cluster labels dictionary (for reference)
cluster_summary = {
    'subject_clusters': {
        str(cid): v for cid, v in subj_cluster_map.items() if cid != -1
    },
    'purpose_clusters': {
        str(cid): v for cid, v in purp_cluster_map.items() if cid != -1
    }
}
with open('cluster_labels.json', 'w') as f:
    json.dump(cluster_summary, f, ensure_ascii=False, indent=2)
print("cluster_labels.json saved")

In [ ]:
import openpyxl
from openpyxl.styles import PatternFill, Alignment, Font, Border, Side

wb = openpyxl.load_workbook(INPUT_XLSX)
ws = wb['Taxonomy']

bold = Font(bold=True)
thin = Side(style='thin', color='CCCCCC')
bdr  = Border(left=thin, right=thin, top=thin, bottom=thin)
wrap = Alignment(wrap_text=True, vertical='top')
FILL = PatternFill('solid', fgColor='E8D5F5')   # Light Purple — distinguish new columns

for col, header in [(COL_SUBJ_CLUSTER, '⑬ Subject Cluster'),
                    (COL_PURP_CLUSTER, '⑭ Purpose Cluster')]:
    cell = ws.cell(1, col, header)
    cell.font   = bold
    cell.fill   = FILL
    cell.border = bdr
    ws.column_dimensions[
        openpyxl.utils.get_column_letter(col)].width = 35

for row_num, data in row_clusters.items():
    for col, text in [(COL_SUBJ_CLUSTER, data['subj_text']),
                      (COL_PURP_CLUSTER, data['purp_text'])]:
        cell = ws.cell(row_num, col, text)
        cell.fill      = FILL
        cell.border    = bdr
        cell.alignment = wrap

wb.save(OUTPUT_XLSX)
print(f"Saved → {OUTPUT_XLSX}")

In [ ]:
files.download(OUTPUT_XLSX)
files.download('cluster_labels.json')